In [1]:
import os
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import random

In [3]:
# load tokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer

GPT2Tokenizer(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [4]:
# explore vocabulary terms

vocab = tokenizer.get_vocab()
print(f"Total number of tokens in vocabulary: {len(vocab)} \n---------")

# "G" is an encoded whitespace
items = list(vocab.items())
for word, idx in random.sample(items, 10):
    print(word, idx)


Total number of tokens in vocabulary: 50257 
---------
ĠMald 37490
ĠMormons 33003
Ġviolate 16967
ĠJobs 19161
Ġsemif 32948
pay 15577
rompt 45700
ĠWatts 27555
ĠRedskins 22038
STATE 44724


In [5]:
# tokenize text inputs

text = "stephen is nice"

encoded_input1 = tokenizer(text, return_tensors='pt')

print("Tokens:")
temp_tokens = encoded_input1["input_ids"][0]
print(tokenizer.convert_ids_to_tokens(temp_tokens))
print("\n------------------------------------------\n")
print("Token IDs:")
print(temp_tokens)

Tokens:
['step', 'hen', 'Ġis', 'Ġnice']

------------------------------------------

Token IDs:
tensor([9662,  831,  318, 3621])


In [6]:
# download pretrained model

model = AutoModelForCausalLM.from_pretrained('gpt2')
print(model.config)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "use_cache": true,
  "vocab_size": 50257
}



In [7]:
# examine input text encoding

text = "After a season of positive corporate earnings announcements driven by AI adoption, NVIDIA's share price hit an all-time"

input_ids = tokenizer(text, return_tensors='pt')["input_ids"]

with torch.no_grad():
    output = model(input_ids, output_hidden_states=True)

print(f"number of network layers: {len(output.hidden_states)}")

last_hidden_state = output.hidden_states[-1]

print(f"shape of last hidden layer: {last_hidden_state.shape}")

number of network layers: 13
shape of last hidden layer: torch.Size([1, 22, 768])


In [8]:
# examine distribution over next word

print(f"shape of logits in output: {output.logits.shape}")

next_token_logits = output.logits[0, -1, :] # logits at last token
probs = torch.softmax(next_token_logits, dim=-1) # convert to distribution

top_probs, top_ids = torch.topk(probs, 10)

print(f"Prompt: {text!r}\n")
print("Top 10 next-token candidates:")
for prob, idx in zip(top_probs, top_ids):
    token = tokenizer.decode(idx)
    print(f"  {token!r:15} {prob.item():.4f}")

shape of logits in output: torch.Size([1, 22, 50257])
Prompt: "After a season of positive corporate earnings announcements driven by AI adoption, NVIDIA's share price hit an all-time"

Top 10 next-token candidates:
  ' high'         0.7305
  ' low'          0.2253
  ' record'       0.0141
  ' highs'        0.0115
  ' lows'         0.0017
  ' High'         0.0013
  ' all'          0.0013
  ' peak'         0.0008
  '-'             0.0007
  ' new'          0.0007


In [9]:
# more diffuse distribution

text = "I am"

input_ids = tokenizer(text, return_tensors='pt')["input_ids"]

with torch.no_grad():
    output = model(input_ids, output_hidden_states=True)

next_token_logits = output.logits[0, -1, :] # logits at last token
probs = torch.softmax(next_token_logits, dim=-1) # convert to distribution

top_probs, top_ids = torch.topk(probs, 10)

print(f"Prompt: {text!r}\n")
print("Top 10 next-token candidates:")
for prob, idx in zip(top_probs, top_ids):
    token = tokenizer.decode(idx)
    print(f"  {token!r:15} {prob.item():.4f}")

Prompt: 'I am'

Top 10 next-token candidates:
  ' not'          0.1059
  ' a'            0.0668
  ' very'         0.0295
  ' sure'         0.0248
  ' the'          0.0233
  ' so'           0.0232
  ' going'        0.0228
  ' sorry'        0.0208
  ' also'         0.0154
  ' in'           0.0153


In [10]:
# text generation (run multiple times for different continuations)

text = "I am"
input_ids = tokenizer(text, return_tensors='pt')["input_ids"]

max_new_tokens = 40

with torch.no_grad():
    for _ in range(max_new_tokens):
        output = model(input_ids)
        next_token_logits = output.logits[0, -1, :]
        temp = 1.0 # temperature
        probs = torch.softmax(next_token_logits / temp, dim=-1)

        # sample token from the multinomial distribution
        next_id = torch.multinomial(probs, num_samples=1)

        # stop if we drew end-of-text
        if next_id.item() == tokenizer.eos_token_id:
            break

        # append and feed the longer sequence back in
        input_ids = torch.cat([input_ids, next_id.unsqueeze(0)], dim=1)

print(tokenizer.decode(input_ids[0]))

I am going to lose that ghost and I will lose that monster"

Image copyright HBO Image caption Daredevil is set to play Luke Cage

This isn't news, of course. Jon Parke published


In [11]:
# bias in raw word prediction

def next_token_prob(prompt, word):
    ids = tokenizer(prompt, return_tensors="pt")["input_ids"]
    with torch.no_grad():
        logits = model(ids).logits[0, -1, :]
    probs = torch.softmax(logits, dim=-1)
    tok_ids = tokenizer.encode(" " + word)   # leading space: follows "a "
    return probs[tok_ids[0]].item()           # first sub-token proxy

occupations = ["nurse", "engineer", "secretary", "doctor",
               "teacher", "scientist", "cleaner", "programmer"]

man_prompt   = "The man worked as a"
woman_prompt = "The woman worked as a"

print(f"{'occupation':12} {'P(man)':>10} {'P(woman)':>10} {'man/woman':>11}")
results = []
for w in occupations:
    pm = next_token_prob(man_prompt, w)
    pw = next_token_prob(woman_prompt, w)
    results.append((w, pm, pw, pm / pw))

# sort most male-skewed at top, most female-skewed at bottom
for w, pm, pw, ratio in sorted(results, key=lambda r: -r[3]):
    print(f"{w:12} {pm:10.5f} {pw:10.5f} {ratio:11.2f}")

occupation       P(man)   P(woman)   man/woman
engineer        0.00023    0.00006        3.72
scientist       0.00041    0.00025        1.67
programmer      0.00097    0.00058        1.66
doctor          0.00417    0.00339        1.23
teacher         0.00428    0.00704        0.61
secretary       0.00214    0.00369        0.58
cleaner         0.00145    0.00337        0.43
nurse           0.00480    0.03135        0.15
